# Analog transmon-controlled SWAP from two detuned $ef$ sidebands

One pulse, two knobs ($\Delta$ and $T$). Self-contained: QuTiP plus the cells below.

## Protocol

States are $|M,q,N\rangle$ = cavity 1 $\otimes$ transmon $\otimes$ cavity 2. Two sidebands on
at once, both on the transmon $e\!-\!f$ transition, both detuned by the same $\Delta$:

| sideband | connects |
|---|---|
| 1 | $\lvert M\!-\!1,f,N\rangle \leftrightarrow \lvert M,e,N\rangle$ |
| 2 | $\lvert M,f,N\!-\!1\rangle \leftrightarrow \lvert M,e,N\rangle$ |

Both terminate on the same $|M,e,N\rangle$. With $f$ above $e$, the two $|f\rangle$ states are
the upper arms and $|M,e,N\rangle$ is the shared **lower vertex** — a V:

```
   |M-1, f, N>                                |M, f, N-1>      upper arms, detuned by Delta
              \                              /
   sideband 1  \                            /  sideband 2
                \------  |M, e, N>  --------/                  shared lower vertex
```

Eliminating the detuned arms leaves two second-order processes:

* **Stark shift of the vertex** (up and back down the same arm) — a dispersive $\chi$;
* **Raman transfer** (up one arm, down the other) — lands on $|M{+}1,e,N{-}1\rangle$, i.e. a
  photon has moved between the cavities: a **conditional beamsplitter**.

They arrive locked in the ratio that makes $\chi(n_1{+}n_2)+J(a_1^\dagger a_2+\text{h.c.})
\propto n_-$ with $a_-=(a_1-a_2)/\sqrt2$, and since $\text{SWAP}=e^{i\pi n_-}$ the gate is a
controlled SWAP. $|g\rangle$ never appears in the ideal Hamiltonian, so the idle branch is
exactly idle.

## Figure of merit

The gate rate is the **effective conditional rate** $\chi_{\rm eff}\sim g^2/\Delta$, so the
merit is the **gate time**

$$T=\frac{\pi}{\chi_{\rm eff}}$$

in real units. Note $g\,T$ is *not* a useful merit here: $T\propto\Delta/g^2$, so a small
$g\,T$ just means driving hard, which is the low-fidelity regime.

## Operating point

Two cavities holding at most **8 photons in total**. That is the self-consistent bound: the
gate contains a real beamsplitter rotation, so a cavity can transiently hold up to $M+N$
photons, and bounding the total by 8 is what keeps every cavity under 8.

## Units

Frequencies in `GHz` where `GHz = 2*pi` rad/ns, so **all times come out in ns**.

### Units and device parameters

A frequency $f$ quoted in GHz means $\omega=2\pi f\times10^{9}\ \mathrm{rad/s}=2\pi f\ \mathrm{rad/ns}$,
so setting `GHz = 2*pi` fixes the time unit to **nanoseconds**.

The transmon is a Kerr oscillator,

$$E_k=k\,\omega_q+\frac{\alpha}{2}k(k-1)$$

In [ ]:
import numpy as np
import qutip as qt
import matplotlib.pyplot as plt

# 1 GHz  ->  omega = 2 pi f = 2 pi rad/ns, so all times below are in ns
GHz = 2 * np.pi
MHz = GHz / 1000

# ---- device parameters ------------------------------------------------------
W_Q = 4.6 * GHz          # transmon g-e frequency
ALPHA = -0.150 * GHz     # anharmonicity
W_1 = 5.1 * GHz          # cavity 1
W_2 = 5.5 * GHz          # cavity 2
N_Q = 4                  # transmon levels kept: g, e, f, h

# ---- operating point -------------------------------------------------------
N_TOT = 8                # max total photons in the two cavities
N_C = N_TOT + 2          # cavity truncation (a cavity never exceeds N_TOT)

# the cavity states the gate is scored on
OP_STATES = []
for M in range(N_TOT + 1):
    for N in range(N_TOT + 1):
        if M + N <= N_TOT:
            OP_STATES.append((M, N))

print(f"transmon: {N_Q} levels, w_q/2pi = {W_Q/GHz:.3f} GHz, "
      f"alpha/2pi = {ALPHA/GHz*1000:+.0f} MHz")
print(f"cavities: {W_1/GHz:.1f} and {W_2/GHz:.1f} GHz")
print(f"operating point: total N <= {N_TOT}  ({len(OP_STATES)} cavity states), "
      f"cutoff N_C = {N_C}")

## 1. Transmon ladder and the static rotating frame

The transmon is a Kerr oscillator, $E_k=k\,\omega_q+\tfrac{\alpha}{2}k(k-1)$, so the rung
frequencies are $\omega_k^{\rm rung}=\omega_q+\alpha k$ for $k\to k+1$.

Each pump is set so that its own cavity's $ef$ sideband is detuned by $\Delta$:

$$\omega_{p_j}=\omega_j-\omega^{\rm rung}_{ef}-\Delta$$

A single **static** rotating frame exists for all the same-cavity sidebands at once. In it
the cavities sit at zero and the transmon level energies must satisfy

$$\varepsilon_{k+1}-\varepsilon_k=\delta_k,\qquad
\delta_k=\omega^{\rm rung}_{ef}-\omega^{\rm rung}_k+\Delta$$

so $\delta_{ef}=\Delta$ by construction, and the neighbouring rungs come out at
$\delta_{ge}=\Delta+\alpha$ and $\delta_{fh}=\Delta-\alpha$ — computed, not assumed.

In [ ]:
def rung_frequency(k):
    # w_k^rung = w_q + alpha * k        (from E_k = k w_q + (alpha/2) k(k-1))
    return W_Q + ALPHA * k


def sideband_detuning(k, Delta):
    # delta_k = w_ef - w_k^rung + Delta,  with the pumps set so that delta_ef = Delta
    return rung_frequency(1) - rung_frequency(k) + Delta


def transmon_frame_energies(Delta):
    # static frame requires  eps_{k+1} - eps_k = delta_k ,  with eps_0 = 0
    energies = [0.0]
    for k in range(N_Q - 1):
        energies.append(energies[-1] + sideband_detuning(k, Delta))
    return energies


RUNG_NAME = {0: "ge", 1: "ef", 2: "fh"}

print("transmon rungs:")
for k in range(N_Q - 1):
    print(f"   {RUNG_NAME[k]}: {rung_frequency(k)/GHz:.3f} GHz")

Delta_demo = 60 * MHz
print(f"\nsideband detunings at Delta/2pi = {Delta_demo/MHz:+.0f} MHz:")
for k in range(N_Q - 1):
    print(f"   {RUNG_NAME[k]}: {sideband_detuning(k, Delta_demo)/MHz:+8.1f} MHz")

print(f"\ncheck: Delta + alpha = {(Delta_demo + ALPHA)/MHz:+.1f} MHz,"
      f"   Delta - alpha = {(Delta_demo - ALPHA)/MHz:+.1f} MHz")

print("\nframe level energies (MHz):",
      [f"{e/MHz:+.1f}" for e in transmon_frame_energies(Delta_demo)])

## 2. The two Hamiltonians

Both in the static frame of section 1. The only difference is which part of the transmon
lowering operator the sideband couples to — nothing is hand-parameterised, the matrix
elements come straight out of `qt.destroy(N_Q)`:

* **ideal model** — only the $ef$ rung is driven. Take the $ef$ matrix element of
  $q=$ `destroy(N_Q)`, i.e. $\langle e|q|f\rangle\,|e\rangle\langle f|$. Then $|g\rangle$ and
  $|h\rangle$ never appear.
* **full model** — the pump couples to the whole operator $q$, so every rung is driven with
  its own matrix element $\langle k|q|k{+}1\rangle=\sqrt{k+1}$, each at its own detuning
  $\delta_k$.

$g$ is the **drive amplitude**, proportional to the RF voltage at the coupler (so RF power
$\propto g^2$). It multiplies the transmon **mode operator** $q$, not any individual
transition, so no matrix element enters $g$ itself — the matrix elements only decide how
strongly each rung responds to that one fixed drive.

**This is what makes the two models comparable**: run them at the same $g$ and you are
sending in the same RF voltage. The $ef$ drive is then *identical* in both, and the full model
simply has extra rungs responding to the same tone. Verified below.

The Hamiltonian is

$$H=\sum_{k}\varepsilon_k\,|k\rangle\langle k|
\;+\;g\Big(\hat D\,\hat q_{\rm drive}+\text{h.c.}\Big)$$

$$\hat D=(1+\epsilon)\,a_1^\dagger+e^{i\Delta\phi}a_2^\dagger,
\qquad
a_\pm=\frac{a_1\pm a_2}{\sqrt2}$$

with $\hat q_{\rm drive}=q$ for the full model and $\hat q_{\rm drive}=\sqrt2\,|e\rangle\langle f|$
(the $ef$ block of $q$) for the ideal one.

At $\Delta\phi=\pi$ and $\epsilon=0$ we have $\hat D=\sqrt2\,a_-^\dagger$, so the individual
rung rates follow from $\langle k|q|k{+}1\rangle=\sqrt{k+1}$:

| rung | supermode coupling |
|---|---|
| $ge$ | $\sqrt2\,g$ |
| $ef$ | $2g$ |
| $fh$ | $\sqrt6\,g$ |

The $ef$ supermode coupling is therefore $G=2g$, and second-order perturbation theory gives

$$\chi_{\rm eff}\approx\frac{G^2}{\Delta}=\frac{4g^2}{\Delta}$$

for the ideal model (where the $|g\rangle$ branch is unshifted, so the whole shift is
conditional).

In [ ]:
def cavity_and_transmon_ops(Nc):
    # a1, a2, and the transmon lowering operator q, in the order c1 (x) Q (x) c2
    Ic = qt.qeye(Nc)
    Iq = qt.qeye(N_Q)
    a1 = qt.tensor(qt.destroy(Nc), Iq, Ic)
    a2 = qt.tensor(Ic, Iq, qt.destroy(Nc))
    q = qt.tensor(Ic, qt.destroy(N_Q), Ic)
    return a1, a2, q


def transmon_level_projector(Nc, k):
    Ic = qt.qeye(Nc)
    return qt.tensor(Ic, qt.basis(N_Q, k) * qt.basis(N_Q, k).dag(), Ic)


def ef_only_lowering(Nc):
    # the ef block of q on its own:  <e|q|f> |e><f|  =  sqrt(2) |e><f|
    Ic = qt.qeye(Nc)
    q_small = qt.destroy(N_Q)
    ef_element = q_small[1, 2]                     # <e|q|f> = sqrt(2)
    ef_op = ef_element * qt.basis(N_Q, 1) * qt.basis(N_Q, 2).dag()
    return qt.tensor(Ic, ef_op, Ic)


def build_H(Nc, g, Delta, model="full", dphi=np.pi, imbalance=0.0):
    a1, a2, q_full = cavity_and_transmon_ops(Nc)

    # sum_k eps_k |k><k|
    energies = transmon_frame_energies(Delta)
    H = 0 * qt.qeye(a1.dims[0])
    for k in range(N_Q):
        H = H + energies[k] * transmon_level_projector(Nc, k)

    # D = (1 + eps) a1^dag + e^{i dphi} a2^dag       (= sqrt(2) a_-^dag at dphi = pi)
    drive = (1 + imbalance) * a1.dag() + np.exp(1j * dphi) * a2.dag()

    # q_drive = q (full)  or  <e|q|f> |e><f| (ideal)
    q_drive = q_full if model == "full" else ef_only_lowering(Nc)

    # H += g ( D q_drive + h.c. )
    exchange = g * (drive * q_drive)
    H = H + exchange + exchange.dag()
    return H


g_demo = 1 * MHz
H_ideal = build_H(N_C, g_demo, Delta_demo, model="ideal")
H_full = build_H(N_C, g_demo, Delta_demo, model="full")

print(f"g/2pi = {g_demo/MHz:.1f} MHz, Delta/2pi = {Delta_demo/MHz:+.0f} MHz")
print(f"ideal model: |g> untouched?  ||H P_g|| = "
      f"{(H_ideal * transmon_level_projector(N_C, 0)).norm():.2e}")
print(f"full model : |g> untouched?  ||H P_g|| = "
      f"{(H_full * transmon_level_projector(N_C, 0)).norm():.2e}")
# supermode coupling of each rung = g * |D| * |<k|q|k+1>| = g * sqrt(2) * sqrt(k+1)
print()
print("rung couplings to the a_- supermode, in units of g:")
for k in range(N_Q - 1):
    element = abs(qt.destroy(N_Q)[k, k + 1])              # <k|q|k+1> = sqrt(k+1)
    print(f"   {RUNG_NAME[k]}:  sqrt(2) * {element:.4f} = {np.sqrt(2)*element:.4f} g")

In [ ]:
# Same g means the same RF voltage.  Check that the ef drive really is identical in
# the two models, so that any difference is purely the extra rungs.
ket = lambda m, q, n: qt.tensor(qt.basis(N_C, m), qt.basis(N_Q, q), qt.basis(N_C, n))
element = lambda H, bra, k: complex(bra.dag() * H * k)

print("at the same g, is the ef coupling <M,e,N| H |M-1,f,N> identical?")
for (M, N) in ((3, 4), (1, 0), (5, 2)):
    left = ket(M, 1, N)
    right = ket(M - 1, 2, N)
    v_ideal = element(H_ideal, left, right)
    v_full = element(H_full, left, right)
    print(f"   <{M},e,{N}| H |{M-1},f,{N}>:  ideal {v_ideal.real:+.6f}"
          f"   full {v_full.real:+.6f}   equal: {np.isclose(v_ideal, v_full)}")

# H_full - H_ideal should live entirely on the ge and fh blocks
difference = H_full - H_ideal
print()
print("norm of (H_full - H_ideal) on each transmon rung block:")
for k in range(N_Q - 1):
    P_lower = transmon_level_projector(N_C, k)
    P_upper = transmon_level_projector(N_C, k + 1)
    block = P_lower * difference * P_upper
    print(f"   {RUNG_NAME[k]}: {block.norm():.4e}")
print()
print("=> the ef drive is untouched; the full model just adds the ge and fh rungs")
print("   at the same RF voltage.")

## 3. Dispersive shifts

Diagonalise each Hamiltonian and read off the shift of every state $|q,n_-\rangle$: find the
eigenvector with the largest overlap on the bare state and take its eigenvalue. Doing it
this way is non-perturbative, so it stays valid where $g^2/\Delta$ expansions start to fail.

The gate needs the shift to be **linear in $n_-$**, and the $e$-minus-$g$ slope is the
conditional rate. The states used are the $a_-$ number states

$$|n\rangle_{a_-}=\frac{(a_-^\dagger)^n}{\sqrt{n!}}\,|0\rangle$$

and the shift of $|q,n\rangle$ is taken as the eigenvalue of the eigenvector with the largest
overlap on it,

$$\lambda_{q,n}=E_{j^\star},\qquad
j^\star=\arg\max_j\big|\langle v_j|q,n\rangle\big|$$

In [ ]:
def n_minus_state(Nc, n, q):
    # |n>_{a_-} (x) |q>  =  (a_-^dag)^n |0,q,0> / sqrt(n!)   (unit() does the 1/sqrt(n!))
    a1, a2, _ = cavity_and_transmon_ops(Nc)
    a_minus = (a1 - a2) / np.sqrt(2)          # a_- = (a1 - a2)/sqrt(2)
    psi = qt.tensor(qt.basis(Nc, 0), qt.basis(N_Q, q), qt.basis(Nc, 0))
    for _ in range(n):
        psi = a_minus.dag() * psi
    return psi.unit()


def dispersive_shifts(H, Nc, levels=(0, 1), n_max=N_TOT):
    # shift of |q, n_-> for each transmon level q and each n_-
    eigenvalues, eigenvectors = H.eigenstates()
    columns = [v.full().ravel() for v in eigenvectors]
    basis_matrix = np.column_stack(columns)

    shifts = {}
    for q in levels:
        row = []
        for n in range(n_max + 1):
            bare = n_minus_state(Nc, n, q).full().ravel()
            # |<v_j | q,n>| for every eigenvector, then take the eigenvalue of the best
            overlaps = np.abs(basis_matrix.conj().T @ bare)
            row.append(np.real(eigenvalues[np.argmax(overlaps)]))
        shifts[q] = np.array(row)
    return shifts


shifts_ideal = dispersive_shifts(H_ideal, N_C)
shifts_full = dispersive_shifts(H_full, N_C)

print(f"dispersive shifts / 2pi in MHz   (g/2pi = {g_demo/MHz:.1f} MHz, "
      f"Delta/2pi = {Delta_demo/MHz:+.0f} MHz)")
print(f"{'n_-':>4} {'ideal g':>10} {'ideal e':>10} {'full g':>10} {'full e':>10}")
for n in range(N_TOT + 1):
    print(f"{n:>4} {shifts_ideal[0][n]/MHz:10.4f} {shifts_ideal[1][n]/MHz:10.4f}"
          f" {shifts_full[0][n]/MHz:10.4f} {shifts_full[1][n]/MHz:10.4f}")

print("\nideal model: the g branch is exactly unshifted, so the whole shift is conditional.")
print("full model : the g branch is shifted too, by the parasitic ge sideband.")

In [ ]:
n_axis = np.arange(N_TOT + 1)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))

axes[0].plot(n_axis, shifts_ideal[0] / MHz, "o-", label="|g> ideal")
axes[0].plot(n_axis, shifts_ideal[1] / MHz, "s-", label="|e> ideal")
axes[0].plot(n_axis, shifts_full[0] / MHz, "o--", label="|g> full")
axes[0].plot(n_axis, shifts_full[1] / MHz, "s--", label="|e> full")
axes[0].set_xlabel(r"$n_-$")
axes[0].set_ylabel(r"shift $/2\pi$  [MHz]")
axes[0].set_title("dispersive shifts")
axes[0].legend(fontsize=8)

diff_ideal = shifts_ideal[1] - shifts_ideal[0]
diff_full = shifts_full[1] - shifts_full[0]
axes[1].plot(n_axis, diff_ideal / MHz, "s-", label="ideal")
axes[1].plot(n_axis, diff_full / MHz, "s--", label="full")
axes[1].set_xlabel(r"$n_-$")
axes[1].set_ylabel(r"(e $-$ g) shift $/2\pi$  [MHz]")
axes[1].set_title("conditional shift: slope is the gate rate")
axes[1].legend(fontsize=8)

for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Effective rate and gate time

The conditional shift is linear in $n_-$; its slope is $\chi_{\rm eff}$, and the gate needs
$\chi_{\rm eff}T=\pi$. Perturbatively $\chi_{\rm eff}\approx G^2/\Delta=4g^2/\Delta$ for the
ideal model; the fitted slope is the number to actually use.

$$\chi_{\rm eff}=\frac{d}{dn_-}\Big(\lambda_{e,n}-\lambda_{g,n}\Big),
\qquad
T=\frac{\pi}{|\chi_{\rm eff}|}$$

In [ ]:
def conditional_rate(H, Nc, n_max=N_TOT):
    # chi_eff = d(lambda_e - lambda_g)/d n_-   (slope of a straight-line fit)
    shifts = dispersive_shifts(H, Nc, n_max=n_max)
    difference = shifts[1] - shifts[0]
    slope, offset = np.polyfit(np.arange(n_max + 1), difference, 1)
    return slope


def gate_time(H, Nc, n_max=N_TOT):
    # T = pi / |chi_eff|
    return np.pi / abs(conditional_rate(H, Nc, n_max))


rate_ideal = conditional_rate(H_ideal, N_C)
rate_full = conditional_rate(H_full, N_C)
T_ideal = np.pi / abs(rate_ideal)
T_full = np.pi / abs(rate_full)

print(f"g/2pi = {g_demo/MHz:.1f} MHz, Delta/2pi = {Delta_demo/MHz:+.0f} MHz")
# chi_eff ~ G^2/Delta with G = 2g  ->  4 g^2 / Delta
print(f"  perturbative 4 g^2 / Delta   = {4*g_demo**2/Delta_demo/MHz:.5f} MHz x 2pi")
print(f"  ideal model chi_eff          = {abs(rate_ideal)/MHz:.5f} MHz x 2pi"
      f"   ->  T = {T_ideal:8.1f} ns = {T_ideal/1000:.3f} us")
print(f"  full  model chi_eff          = {abs(rate_full)/MHz:.5f} MHz x 2pi"
      f"   ->  T = {T_full:8.1f} ns = {T_full/1000:.3f} us")

In [ ]:
print("rate and gate time versus detuning   (g/2pi = 1 MHz)")
print(f"{'D/2pi MHz':>10} {'ideal MHz':>11} {'T ideal us':>12} "
      f"{'full MHz':>10} {'T full us':>11}")

detuning_list = [-140, -120, -90, -60, -40, 40, 60, 90, 120, 140]
rate_table = []
for D_MHz in detuning_list:
    D = D_MHz * MHz
    Hi = build_H(N_C, g_demo, D, model="ideal")
    Hf = build_H(N_C, g_demo, D, model="full")
    ri = abs(conditional_rate(Hi, N_C))
    rf = abs(conditional_rate(Hf, N_C))
    rate_table.append((D_MHz, ri, rf))
    print(f"{D_MHz:>10} {ri/MHz:11.5f} {np.pi/ri/1000:12.3f} "
          f"{rf/MHz:10.5f} {np.pi/rf/1000:11.3f}", flush=True)

In [ ]:
D_axis = np.array([row[0] for row in rate_table])
T_ideal_axis = np.array([np.pi / row[1] / 1000 for row in rate_table])
T_full_axis = np.array([np.pi / row[2] / 1000 for row in rate_table])

fig, ax = plt.subplots(figsize=(7, 4))
negative = D_axis < 0
ax.plot(D_axis[negative], T_ideal_axis[negative], "o-", color="C0", label="ideal ($ef$ only)")
ax.plot(D_axis[negative], T_full_axis[negative], "s-", color="C1", label="full transmon")
ax.plot(D_axis[~negative], T_ideal_axis[~negative], "o-", color="C0")
ax.plot(D_axis[~negative], T_full_axis[~negative], "s-", color="C1")
ax.set_xlabel(r"$\Delta/2\pi$  [MHz]")
ax.set_ylabel(r"gate time $T=\pi/\chi_{\rm eff}$  [$\mu$s]")
ax.set_title(f"gate time at $g/2\\pi$ = {g_demo/MHz:.0f} MHz")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. The controlled SWAP in action

Now watch it work. Start from $|3,q,0\rangle$ — three photons in cavity 1, none in cavity 2 —
and follow $\langle n_1\rangle$ and $\langle n_2\rangle$ through the gate.

Expected: on the $|e\rangle$ branch the photons move across, $3\to0$ and $0\to3$; on the
$|g\rangle$ branch nothing happens. The $|f\rangle$ population should stay small throughout,
since the arms of the V are only virtually occupied.

$$|\psi(t)\rangle=e^{-iHt}|\psi(0)\rangle,\qquad
\langle n_j\rangle=\langle\psi(t)|a_j^\dagger a_j|\psi(t)\rangle,\qquad
P_k=\langle\psi(t)|\,\mathbb 1\otimes|k\rangle\langle k|\otimes\mathbb 1\,|\psi(t)\rangle$$

In [ ]:
def evolve_populations(H, Nc, M0, N0, q0, T, steps=60):
    # <n1>, <n2>, P_f and P_h versus time, starting from |M0, q0, N0>
    a1, a2, _ = cavity_and_transmon_ops(Nc)
    n1_op = a1.dag() * a1
    n2_op = a2.dag() * a2
    Pf_op = transmon_level_projector(Nc, 2)
    Ph_op = transmon_level_projector(Nc, 3)

    psi = qt.tensor(qt.basis(Nc, M0), qt.basis(N_Q, q0), qt.basis(Nc, N0))
    U_step = (-1j * H * (T / steps)).expm()      # exp(-i H dt), applied repeatedly

    times, n1, n2, pf, ph = [], [], [], [], []
    for step in range(steps + 1):
        times.append(step * T / steps)
        n1.append(qt.expect(n1_op, psi).real)
        n2.append(qt.expect(n2_op, psi).real)
        pf.append(qt.expect(Pf_op, psi).real)
        ph.append(qt.expect(Ph_op, psi).real)
        psi = U_step * psi
    return np.array(times), np.array(n1), np.array(n2), np.array(pf), np.array(ph)


t_e, n1_e, n2_e, pf_e, ph_e = evolve_populations(H_ideal, N_C, 3, 0, 1, T_ideal)
t_g, n1_g, n2_g, pf_g, ph_g = evolve_populations(H_ideal, N_C, 3, 0, 0, T_ideal)

print("ideal model, input |3, q, 0>")
print(f"   transmon in |e>:  <n1> {n1_e[0]:.3f} -> {n1_e[-1]:.3f},"
      f"   <n2> {n2_e[0]:.3f} -> {n2_e[-1]:.3f}")
print(f"   transmon in |g>:  <n1> {n1_g[0]:.3f} -> {n1_g[-1]:.3f},"
      f"   <n2> {n2_g[0]:.3f} -> {n2_g[-1]:.3f}")
print(f"   max P_f on the e branch: {pf_e.max():.4f}   (virtual, as expected)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))

axes[0].plot(t_e / 1000, n1_e, label=r"$\langle n_1\rangle$")
axes[0].plot(t_e / 1000, n2_e, label=r"$\langle n_2\rangle$")
axes[0].set_title(r"control $|e\rangle$: photons swap")
axes[0].legend()

axes[1].plot(t_g / 1000, n1_g, label=r"$\langle n_1\rangle$")
axes[1].plot(t_g / 1000, n2_g, label=r"$\langle n_2\rangle$")
axes[1].set_title(r"control $|g\rangle$: nothing happens")
axes[1].legend()

axes[2].plot(t_e / 1000, pf_e, label=r"$P_f$, control $|e\rangle$")
axes[2].plot(t_g / 1000, pf_g, label=r"$P_f$, control $|g\rangle$")
axes[2].set_title("the V arms stay virtual")
axes[2].legend()

for ax in axes:
    ax.set_xlabel(r"time  [$\mu$s]")
    ax.set_ylim(bottom=-0.05)
    ax.grid(alpha=0.3)
axes[0].set_ylabel("photon number")
axes[2].set_ylabel(r"$P_f$")
plt.suptitle(f"ideal model, input $|3,q,0\\rangle$, "
             f"T = {T_ideal/1000:.2f} $\\mu$s", y=1.02)
plt.tight_layout()
plt.show()

### The same thing with the full transmon

Here the $|g\rangle$ branch drifts, because the parasitic $ge$ sideband shifts it too. Only
the *difference* between the branches is calibrated to $\pi$ — which is exactly why the gate
time came from the $e$-minus-$g$ slope in section 4. The residual common rotation is an
unconditional beamsplitter, which one compensating cavity–cavity beamsplitter removes.

In [ ]:
t_ef, n1_ef, n2_ef, pf_ef, ph_ef = evolve_populations(H_full, N_C, 3, 0, 1, T_full)
t_gf, n1_gf, n2_gf, pf_gf, ph_gf = evolve_populations(H_full, N_C, 3, 0, 0, T_full)

print("full model, input |3, q, 0>")
print(f"   transmon in |e>:  <n1> {n1_ef[0]:.3f} -> {n1_ef[-1]:.3f}")
print(f"   transmon in |g>:  <n1> {n1_gf[0]:.3f} -> {n1_gf[-1]:.3f}"
      f"   <- drifts: the g branch is shifted too")
print(f"   max P_f = {pf_ef.max():.4f},  max P_h = {ph_ef.max():.2e}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(t_ef / 1000, n1_ef, label=r"$\langle n_1\rangle$")
axes[0].plot(t_ef / 1000, n2_ef, label=r"$\langle n_2\rangle$")
axes[0].set_title(r"full transmon, control $|e\rangle$")
axes[1].plot(t_gf / 1000, n1_gf, label=r"$\langle n_1\rangle$")
axes[1].plot(t_gf / 1000, n2_gf, label=r"$\langle n_2\rangle$")
axes[1].set_title(r"full transmon, control $|g\rangle$")
for ax in axes:
    ax.set_xlabel(r"time  [$\mu$s]")
    ax.set_ylabel("photon number")
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. What is really happening: both branches rotate

The plot above is not a clean SWAP-versus-nothing, and it should not be. In the full model
the parasitic $ge$ sideband shifts the $|g\rangle$ branch too, so **both** branches undergo a
beamsplitter rotation — just at different rates. What the gate time is calibrated to is the
**difference**:

$$\theta_q=\text{slope}_q\times T,
\qquad
\theta_e-\theta_g=(\chi_e-\chi_g)\,T=\pi$$

Running out to several gate times makes this obvious: the $e$ branch swaps back and forth
quickly, the $g$ branch drifts slowly, and at $t=T$ their accumulated angles differ by
exactly $\pi$. The leftover common rotation is an unconditional beamsplitter — one
compensating cavity–cavity beamsplitter turns this into a clean SWAP-versus-identity, which
is what section 7 does.

In [ ]:
shifts_f = dispersive_shifts(H_full, N_C)
slope_g, offset_g = np.polyfit(n_axis, shifts_f[0], 1)
slope_e, offset_e = np.polyfit(n_axis, shifts_f[1], 1)

print(f"full model, g/2pi = {g_demo/MHz:.1f} MHz, Delta/2pi = {Delta_demo/MHz:+.0f} MHz,"
      f"  T = {T_full/1000:.3f} us")
print(f"  g branch rotates by {slope_g*T_full/np.pi:+.4f} pi  during T")
print(f"  e branch rotates by {slope_e*T_full/np.pi:+.4f} pi  during T")
print(f"  difference          {(slope_e-slope_g)*T_full/np.pi:+.4f} pi"
      f"   <- this is what T is calibrated to")


def trace_n1(H, Nc, M0, N0, q0, t_max, steps=180):
    # <n1> versus time from |M0, q0, N0>
    a1, a2, _ = cavity_and_transmon_ops(Nc)
    n1_op = a1.dag() * a1
    psi = qt.tensor(qt.basis(Nc, M0), qt.basis(N_Q, q0), qt.basis(Nc, N0))
    U_step = (-1j * H * (t_max / steps)).expm()
    times = []
    values = []
    for step in range(steps + 1):
        times.append(step * t_max / steps)
        values.append(qt.expect(n1_op, psi).real)
        psi = U_step * psi
    return np.array(times), np.array(values)


t_long = 3 * T_full
t_le, n1_le = trace_n1(H_full, N_C, 3, 0, 1, t_long)
t_lg, n1_lg = trace_n1(H_full, N_C, 3, 0, 0, t_long)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t_le / 1000, n1_le, label=r"control $|e\rangle$")
ax.plot(t_lg / 1000, n1_lg, label=r"control $|g\rangle$")
for k in (1, 2, 3):
    ax.axvline(k * T_full / 1000, color="k", ls=":", lw=0.8)
ax.text(T_full / 1000, 3.05, " T", fontsize=9)
ax.set_xlabel(r"time  [$\mu$s]")
ax.set_ylabel(r"$\langle n_1\rangle$")
ax.set_title("full transmon: both branches rotate, at different rates")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Fidelity

The target is $U_{\rm target}=\Pi_g\otimes\mathbb 1+\Pi_e\otimes\text{SWAP}$.

Two corrections are free and are applied before scoring:

1. **A virtual Z on each control level** — absorbed by taking the modulus of each branch's
   amplitude separately.
2. **One unconditional cavity–cavity beamsplitter**, to undo the common rotation identified
   in section 6. Its angle is not searched for: the $g$-branch shift is *exactly* linear in
   $n_-$ (see the section-3 table), so the required angle is just
   $\lambda=\text{slope}_g\times T$, read straight off the fit.

For the ideal model $\text{slope}_g=0$, so no compensation is needed at all. For the full
model it is essential — without it the raw gate scores $F\approx0.13$.

$$U_{\rm target}=\Pi_g\otimes\mathbb 1+\Pi_e\otimes\mathrm{SWAP},
\qquad
W=e^{+i\lambda n_-},\quad \lambda=\text{slope}_g\times T$$

$$A_q=\sum_{(M,N)}\big\langle M,q,N\big|\,U_{\rm target}^\dagger\,W\,U\,\big|M,q,N\big\rangle,
\qquad
F=\frac{\big(|A_e|+|A_g|\big)^2}{(2d)^2},\quad d=\#\{(M,N)\}$$

Taking $|A_e|$ and $|A_g|$ separately is what makes the virtual Z on each control level
free.

In [ ]:
def flat_index(Nc, M, q, N):
    # position of |M, q, N> in the flattened basis
    return (M * N_Q + q) * Nc + N


def ideal_controlled_swap(Nc):
    # U_target = P_g (x) 1 + P_e (x) SWAP :  |M,e,N> -> |N,e,M>,  |M,g,N> -> |M,g,N>
    dim = Nc * N_Q * Nc
    U = np.zeros((dim, dim))
    for M in range(Nc):
        for N in range(Nc):
            for q in range(N_Q):
                column = flat_index(Nc, M, q, N)
                if q == 1:
                    row = flat_index(Nc, N, q, M)
                else:
                    row = column
                U[row, column] = 1.0
    return qt.Qobj(U, dims=[[Nc, N_Q, Nc]] * 2)


def compensating_beamsplitter(H, Nc, T):
    # lambda = slope_g * T, then  W = exp(+i lambda n_-)
    shifts = dispersive_shifts(H, Nc)
    slope, offset = np.polyfit(np.arange(N_TOT + 1), shifts[0], 1)   # slope_g
    a1, a2, _ = cavity_and_transmon_ops(Nc)
    a_minus = (a1 - a2) / np.sqrt(2)
    n_minus = a_minus.dag() * a_minus                                # n_- = a_-^dag a_-
    return (1j * slope * T * n_minus).expm()                         # exp(+i lambda n_-)


def gate_fidelity(U, Nc, states):
    # M = U_target^dag U
    M = (ideal_controlled_swap(Nc).dag() * U).full()
    # A_q = sum_{(M,N)} <M,q,N| M |M,q,N>
    amp_e = 0j
    amp_g = 0j
    for (M0, N0) in states:
        index_e = flat_index(Nc, M0, 1, N0)
        index_g = flat_index(Nc, M0, 0, N0)
        amp_e = amp_e + M[index_e, index_e]
        amp_g = amp_g + M[index_g, index_g]
    # F = (|A_e| + |A_g|)^2 / (2d)^2   -- separate moduli => free virtual Z per branch
    d = len(states)
    return (abs(amp_e) + abs(amp_g)) ** 2 / (2 * d) ** 2


def run_gate(g, Delta, model, Nc=N_C):
    # build, calibrate T, propagate, compensate, score
    H = build_H(Nc, g, Delta, model=model)
    T = gate_time(H, Nc)
    U = (-1j * H * T).expm()
    W = compensating_beamsplitter(H, Nc, T)
    return T, gate_fidelity(U, Nc, OP_STATES), gate_fidelity(W * U, Nc, OP_STATES)


print(f"total N <= {N_TOT}, Delta/2pi = {Delta_demo/MHz:+.0f} MHz")
print(f"{'model':>7} {'g/2pi':>7} {'T (us)':>9} {'F raw':>10} {'F compensated':>15}")
for model in ("ideal", "full"):
    for g_MHz in (2.0, 1.0, 0.5):
        T, F_raw, F_comp = run_gate(g_MHz * MHz, Delta_demo, model)
        print(f"{model:>7} {g_MHz:7.2f} {T/1000:9.3f} {F_raw:10.6f} {F_comp:15.8f}",
              flush=True)

### The compensated gate really is a SWAP

With the compensating beamsplitter applied, the $|e\rangle$ branch completes the swap and the
$|g\rangle$ branch is returned to the identity.

In [ ]:
H_c = build_H(N_C, g_demo, Delta_demo, model="full")
T_c = gate_time(H_c, N_C)
U_c = (-1j * H_c * T_c).expm()
W_c = compensating_beamsplitter(H_c, N_C, T_c)

a1_c, a2_c, _ = cavity_and_transmon_ops(N_C)
n1_c = a1_c.dag() * a1_c
n2_c = a2_c.dag() * a2_c

print("input |3, q, 0>, full model, after gate + compensating beamsplitter")
for q, label in ((1, "e"), (0, "g")):
    psi_in = qt.tensor(qt.basis(N_C, 3), qt.basis(N_Q, q), qt.basis(N_C, 0))
    psi_raw = U_c * psi_in
    psi_out = W_c * psi_raw
    print(f"   control |{label}>:  raw <n1> = {qt.expect(n1_c, psi_raw).real:.4f}"
          f"   ->  compensated <n1> = {qt.expect(n1_c, psi_out).real:.4f},"
          f"  <n2> = {qt.expect(n2_c, psi_out).real:.4f}")

## 8. Fidelity and gate time versus detuning

Both models, at fixed drive strength. The ideal model improves steadily with $|\Delta|$ at
the cost of a longer gate. The full model does not: the parasitic rungs sit at
$\Delta+\alpha$ and $\Delta-\alpha$, and one of them goes resonant at $|\Delta|=|\alpha|=150$
MHz, so there is an optimum well inside that.

In [ ]:
print(f"g/2pi = {g_demo/MHz:.1f} MHz, total N <= {N_TOT}")
print(f"{'D/2pi':>7} {'T ideal us':>12} {'1-F ideal':>12} "
      f"{'T full us':>11} {'1-F full':>11}")

fid_table = []
for D_MHz in detuning_list:
    D = D_MHz * MHz
    T_i, raw_i, F_i = run_gate(g_demo, D, "ideal")
    T_f, raw_f, F_f = run_gate(g_demo, D, "full")
    fid_table.append((D_MHz, T_i, 1 - F_i, T_f, 1 - F_f))
    print(f"{D_MHz:+7.0f} {T_i/1000:12.3f} {1-F_i:12.2e} "
          f"{T_f/1000:11.3f} {1-F_f:11.2e}", flush=True)

In [ ]:
D_ax = np.array([row[0] for row in fid_table])
Ti_ax = np.array([row[1] / 1000 for row in fid_table])
Ii_ax = np.array([row[2] for row in fid_table])
Tf_ax = np.array([row[3] / 1000 for row in fid_table])
If_ax = np.array([row[4] for row in fid_table])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(D_ax, Ii_ax, "o-", label="ideal")
axes[0].plot(D_ax, If_ax, "s-", label="full transmon")
axes[0].set_yscale("log")
axes[0].set_xlabel(r"$\Delta/2\pi$  [MHz]")
axes[0].set_ylabel(r"$1-F$")
axes[0].set_title("infidelity")

axes[1].plot(Ti_ax, Ii_ax, "o-", label="ideal")
axes[1].plot(Tf_ax, If_ax, "s-", label="full transmon")
axes[1].set_yscale("log")
axes[1].set_xlabel(r"gate time $T$  [$\mu$s]")
axes[1].set_ylabel(r"$1-F$")
axes[1].set_title("the actual trade-off")

for ax in axes:
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 9. How the error scales with drive strength

The two models behave qualitatively differently, which is worth knowing before trying to
extrapolate anything.

* **Full model** — smooth, $1-F\propto g^{2}$. The parasitic $ge$ shift is a fixed
  *fraction* of the wanted one ($\chi_g/\chi_e\to\tfrac12$ at large $\Delta$), so weakening
  the drive helps only through the leftover higher-order terms.
* **Ideal model** — *oscillatory*, not a power law. The only error is leakage into the
  virtual $|f\rangle$ arms, and that leakage periodically returns to zero. At the revival
  points the infidelity drops by two orders of magnitude. So for the ideal model one tunes
  onto a revival rather than simply driving weaker.

The exponent below is the least-squares slope of

$$\log(1-F)=p\,\log g+\text{const}$$

In [ ]:
g_scan = np.linspace(0.6, 1.4, 17)
inf_ideal = []
inf_full = []
for g_MHz in g_scan:
    T_i, raw_i, F_i = run_gate(g_MHz * MHz, Delta_demo, "ideal")
    T_f, raw_f, F_f = run_gate(g_MHz * MHz, Delta_demo, "full")
    inf_ideal.append(1 - F_i)
    inf_full.append(1 - F_f)
inf_ideal = np.array(inf_ideal)
inf_full = np.array(inf_full)

# p from  log(1-F) = p log g + const
power_full = np.polyfit(np.log(g_scan), np.log(inf_full), 1)[0]
print(f"full model:  1-F ~ g^{power_full:.2f}   (smooth)")
print(f"ideal model: infidelity ranges {inf_ideal.min():.1e} to {inf_ideal.max():.1e}"
      f"  over the same span  (oscillatory)")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(g_scan, inf_ideal, "o-", label="ideal ($f$-leakage revivals)")
ax.plot(g_scan, inf_full, "s-", label="full transmon ($\\propto g^2$)")
ax.set_yscale("log")
ax.set_xlabel(r"$g/2\pi$  [MHz]")
ax.set_ylabel(r"$1-F$")
ax.set_title(f"error vs drive strength at $\\Delta/2\\pi$ = {Delta_demo/MHz:+.0f} MHz")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()